In [ ]:
# OpenCode / OpenAI-compatible notebook setup.
# Install once in the notebook's Python environment: pip install openai python-dotenv pyyaml
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import os as _nb_os

from pathlib import Path as _NotebookPath

from dotenv import load_dotenv as _notebook_load_dotenv
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Paths
#
CHAPTER = "s13_agent_teams"

PROJECT_ROOT = next(
    (
        directory
        for start in [_NotebookPath.cwd(), *_NotebookPath.cwd().parents]
        for directory in [start]
        if ((directory / CHAPTER / "code.py").is_file())
    ),
    None,
)

if (PROJECT_ROOT is None):

    raise (FileNotFoundError(
        f"Start Jupyter from the project root or {CHAPTER}/ "
        "so code.py can be located"
    ))
#

_notebook_load_dotenv(PROJECT_ROOT / ".env", override=True)
_nb_os.chdir(PROJECT_ROOT)

__file__ = str(PROJECT_ROOT / CHAPTER / "code.py")
#
###################################################################################################
###################################################################################################
################################################################################################### Interactive Entry Point
#
# Run All must not start a blocking CLI. Enable this explicitly if needed.
RUN_AGENT_INTERACTIVELY = False
#
###################################################################################################
###################################################################################################
################################################################################################### OpenCode Configuration
#
# The existing ANTHROPIC_* environment names remain supported.
_notebook_key   = (_nb_os.getenv("OPENCODE_API_KEY")  or  _nb_os.getenv("ANTHROPIC_API_KEY"))
_notebook_model = (_nb_os.getenv("OPENCODE_MODEL")  or  _nb_os.getenv("MODEL_ID"))
_notebook_url   = (
    _nb_os.getenv("OPENCODE_BASE_URL")
    or  _nb_os.getenv("ANTHROPIC_BASE_URL")
    or  "https://opencode.ai/inference/openai/v1"
).rstrip("/")

if (_notebook_url.endswith("/chat/completions")):

    _notebook_url = _notebook_url[:-len("/chat/completions")]
#

if (not _notebook_key):

    raise (RuntimeError(
        "Set OPENCODE_API_KEY (or ANTHROPIC_API_KEY) in the root .env"
    ))
#

if (not _notebook_model):

    raise (RuntimeError(
        "Set MODEL_ID (or OPENCODE_MODEL) in the root .env"
    ))
#

_nb_os.environ["OPENCODE_API_KEY"]   = _notebook_key
_nb_os.environ["OPENCODE_BASE_URL"]  = _notebook_url
_nb_os.environ["ANTHROPIC_BASE_URL"] = _notebook_url
_nb_os.environ["MODEL_ID"]          = _notebook_model

print(f"{CHAPTER}: OpenCode via OpenAI-compatible API, model={_notebook_model}")
#


In [ ]:
# Bridge the lesson's Anthropic-style message interface to OpenCode's OpenAI API.
# Legacy chapter loaders import "anthropic" dynamically; this mapping serves those
# imports inside the current notebook kernel without changing the .py files.
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import json  as _compat_json
import sys   as _compat_sys
import types as _compat_types

from openai import OpenAI
import uuid
#
###################################################################################################
###################################################################################################
################################################################################################### Content Conversion
#
def _compat_field(block:object, key:str, default=None):

    """
        Read a response field from either a plain dictionary or provider object.

        The chapter code consumes both forms; this helper keeps those access
        patterns equivalent without mutating the original response.
    """

    return ((block.get(key, default))
            if (isinstance(block, dict))
            else (getattr(block, key, default)))
#


def _compat_text(value:object) -> str:

    """
        Convert Anthropic-style content blocks to Chat Completions text.

        Exclude tool-use blocks: their identifiers and arguments are translated
        separately so that subsequent tool results can be matched correctly.
    """

    if (isinstance(value, str)):

        return (value)
    #
    if (value is None):

        return ("")
    #
    if (isinstance(value, list)):

        return ("\n".join(
            _compat_text(_compat_field(item, "text", _compat_field(item, "content", "")))
            for item in value
            if (_compat_field(item, "type") != "tool_use")
        ))
    #
    if (isinstance(value, dict)):

        return (_compat_text(value.get(
            "text", value.get("content", _compat_json.dumps(value))
        )))
    #
    return (str(value))
#


def _compat_history(messages:list, system:str|list|None=None) -> list:

    """
        Translate chapter conversation history for OpenAI Chat Completions.

        Preserve every assistant tool-call ID and pair it with the corresponding
        tool response. The system prompt is inserted before conversational turns.
    """

    result = []

    if (system):

        result.append({"role": "system", "content": _compat_text(system)})
    #

    for message in messages:

        role    = _compat_field(message, "role")
        content = _compat_field(message, "content", "")

        if (role == "assistant"):

            if (not isinstance(content, list)):

                result.append({"role": "assistant", "content": _compat_text(content)})
                continue
            #

            text_blocks = []
            tool_calls  = []

            for block in content:

                kind = _compat_field(block, "type")

                if (kind == "tool_use"):

                    arguments = _compat_field(block, "input", {})

                    tool_calls.append({
                        "id"       : _compat_field(block, "id"),
                        "type"     : "function",
                        "function" : {
                            "name"      : _compat_field(block, "name"),
                            "arguments" : _compat_json.dumps(arguments, ensure_ascii=False),
                        },
                    })
                #
                elif (kind == "text"):

                    text_blocks.append(_compat_field(block, "text", ""))
                #
            #

            converted = {"role": "assistant", "content": "\n".join(text_blocks) or None}

            if (tool_calls):

                converted["tool_calls"] = tool_calls
            #
            if (converted["content"] is None  and  not tool_calls):

                converted["content"] = ""
            #

            result.append(converted)
        #
        elif (role == "user"  and  isinstance(content, list)):

            user_text    = []
            tool_results = []

            for block in content:

                if (_compat_field(block, "type") == "tool_result"):

                    tool_results.append({
                        "role"         : "tool",
                        "tool_call_id" : _compat_field(block, "tool_use_id"),
                        "content"      : _compat_text(_compat_field(block, "content", "")),
                    })
                #
                else:

                    user_text.append(_compat_text(block))
                #
            #

            result.extend(tool_results)

            if (user_text):

                result.append({"role": "user", "content": "\n".join(user_text)})
            #
        #
        else:

            result.append({"role": role, "content": _compat_text(content)})
        #
    #

    return (result)
#
###################################################################################################
###################################################################################################
################################################################################################### OpenAI Message Adapter
#
class _NotebookMessages:

    """
        Expose messages.create on an OpenAI-compatible chat-completions client.

        Construct function-tool definitions, convert responses to the chapter's
        expected block objects, and translate token counts and stop reasons.
    """

    def __init__(self, client) -> None:

        """
            Keep the provider client used by subsequent model requests.
        """

        self._client = client
    #
    #################################################################  Public API
    #
    def create(
            self,
            *,
            model      : str,
            messages   : list,
            system     : str|list|None = None,
            tools      : list|None     = None,
            max_tokens : int|None      = None,
            **kwargs,
        ) -> object:

        """
            Send an OpenAI-compatible request and return Anthropic-shaped blocks.

            Tool requests remain associated with their call identifiers.
            The translation preserves the original lesson's model/tool loop.
        """

        request = {"model": model, "messages": _compat_history(messages, system)}

        if (tools):

            request["tools"] = [
                {
                    "type"     : "function",
                    "function" : {
                        "name"        : tool["name"],
                        "description" : tool.get("description", ""),
                        "parameters"  : tool.get(
                            "input_schema", {"type": "object", "properties": {}}
                        ),
                    },
                }
                for tool in tools
            ]
        #
        if (max_tokens is not None):

            request["max_tokens"] = max_tokens
        #

        response = self._client.chat.completions.create(**request)
        choice   = response.choices[0]
        reply    = choice.message
        blocks   = []

        if (reply.content):

            blocks.append(_compat_types.SimpleNamespace(
                type = "text",
                text = reply.content,
            ))
        #

        for call in (reply.tool_calls or []):

            try:

                arguments = _compat_json.loads(call.function.arguments or "{}")

                if (not isinstance(arguments, dict)):

                    arguments = {}
                #
            #
            except (ValueError, TypeError):

                arguments = {}
            #

            blocks.append(_compat_types.SimpleNamespace(
                type  = "tool_use",
                id    = call.id,
                name  = call.function.name,
                input = arguments,
            ))
        #

        usage       = getattr(response, "usage", None)
        stop_reason = (
            ("max_tokens") if (choice.finish_reason == "length")
            else (("tool_use") if (reply.tool_calls) else ("end_turn"))
        )

        return (_compat_types.SimpleNamespace(
            content     = blocks,
            stop_reason = stop_reason,
            usage       = _compat_types.SimpleNamespace(
                input_tokens  = getattr(usage, "prompt_tokens", 0)  or  0,
                output_tokens = getattr(usage, "completion_tokens", 0)  or  0,
            ),
        ))
    #
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Client
#
class _NotebookAnthropic:

    """
        Provide a legacy chapter client backed by the OpenCode endpoint.

        The name is retained for import compatibility; requests are always sent
        through the OpenAI Python SDK and use the notebook's configured API key.
    """

    def __init__(self, base_url:str|None=None, **kwargs) -> None:

        """
            Initialize the OpenAI client and expose the message adapter.
        """

        url = (_nb_os.getenv("OPENCODE_BASE_URL")  or  base_url  or  _notebook_url)

        self._client = OpenAI(
            api_key         = _notebook_key,
            base_url        = url,
            default_headers = {
                "x-opencode-session" : f"ses_{uuid.uuid4().hex}",
                "User-Agent"        : "learn-claude-code/1.0",
            },
        )
        self.messages = _NotebookMessages(self._client)
    #
#


_anthropic_compat           = _compat_types.ModuleType("anthropic")
_anthropic_compat.Anthropic = _NotebookAnthropic
_compat_sys.modules["anthropic"] = _anthropic_compat
#


In [ ]:
#!/usr/bin/env python3
"""
s13: Agent Teams - persistent teammates with shared tasks and mailboxes.

Run:  python s13_agent_teams/code.py
Need: pip install anthropic python-dotenv + .env with ANTHROPIC_API_KEY

    +------+  spawn(task_id)  +----------+  result  +------+
    | Lead | ---------------> |   WORK   | -------> | IDLE |
    +--+---+                  +----+-----+          +--+---+
       ^                           |                   |
       | team events               | tools             | wait
       |                           v                   v
    +--+-----------+          +----------+        +----------+
    | MessageBus   |          | Task cwd | <----- | Mailbox  |
    +--------------+          +----------+  claim +----------+

    .tasks/       shared task records and dependencies
    .mailboxes/   messages, results, and protocol responses
    .worktrees/   optional task-bound working directories
"""


In [ ]:

#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import fcntl
import json
import os
import random
import re
import secrets
import select
import subprocess
import sys
import threading
import time
from contextlib import contextmanager
from dataclasses import dataclass, asdict, field
from pathlib import Path

try:

    import readline
    readline.parse_and_bind('set bind-tty-special-chars off')
#
except ImportError:

    pass
#

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
if (os.getenv("ANTHROPIC_BASE_URL")):

    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)
#

WORKDIR = Path.cwd()
client  = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL   = os.environ["MODEL_ID"]


In [ ]:

#
###################################################################################################
###################################################################################################
################################################################################################### Task System
#

TASKS_DIR         = WORKDIR / ".tasks"
TASKS_ROOT        = TASKS_DIR.resolve()
TASK_ID_PATTERN   = re.compile(r"^task_[0-9a-f]{8}$")
task_lock         = threading.RLock()
TASK_LOCK_PATH    = TASKS_DIR / ".lock"
_task_store_state = threading.local()

# owner -> {"task_id": str, "cwd": Path}. A teammate gets one assignment at
# a time, and every filesystem tool resolves its cwd through this registry.
teammate_assignments : dict[str, dict[str, object]] = {}
assignment_versions  : dict[str, int]               = {}


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### Public API
#
@contextmanager
def task_store_lock():

    """Serialize task mutations across threads and host processes."""
    with task_lock:

        depth = getattr(_task_store_state, "depth", 0)
        if (depth == 0):

            TASKS_DIR.mkdir(parents=True, exist_ok=True)
            handle = TASK_LOCK_PATH.open("a+", encoding="utf-8")
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX)
            _task_store_state.handle = handle
        #
        _task_store_state.depth = depth + 1
        try:

            yield
        #
        finally:

            _task_store_state.depth -= 1
            if (_task_store_state.depth == 0):

                handle = _task_store_state.handle
                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)
                handle.close()
                del _task_store_state.handle
            #
        #
    #
#


def advance_assignment_version(owner:str):

    """Invalidate old approvals without clearing an explicit plan requirement."""
    with task_lock:

        assignment_versions[owner] = assignment_versions.get(owner, 0) + 1
        gates                      = globals().get("plan_gates")
        request_ids                = globals().get("plan_request_ids")
        team                       = globals().get("team_lock")
        if (team is not None):

            team.acquire()
        #
        try:

            if (isinstance(gates, dict)  and  owner in gates
                    and  gates[owner] != "not_required"):

                gates[owner] = "required"
            #
            if (isinstance(request_ids, dict)):

                request_ids.pop(owner, None)
            #
        #
        finally:

            if (team is not None):

                team.release()
            #
        #
    #
#


@dataclass
class Task:

    id          : str
    subject     : str
    description : str
    status      : str          # pending | in_progress | completed
    owner       : str|None
    blockedBy   : list[str]
    worktree    : str|None  = None
#


def _task_path(task_id:str) -> Path:

    if (not isinstance(task_id, str)  or  not TASK_ID_PATTERN.fullmatch(task_id)):

        raise (ValueError(f"Invalid task ID: {task_id!r}"))
    #
    path = (TASKS_DIR / f"{task_id}.json").resolve()
    if (not TASKS_ROOT.is_relative_to(WORKDIR.resolve())
            or  not path.is_relative_to(TASKS_ROOT)):

        raise (ValueError(f"Invalid task ID: {task_id!r}"))
    #
    return (path)
#


def create_task(subject:str, description:str="") -> Task:

    subject = subject.strip()
    if (not subject):

        raise (ValueError("Task subject cannot be empty"))
    #
    with task_store_lock():

        for _ in range(100):

            task = Task(
                id          = f"task_{secrets.token_hex(4)}",
                subject     = subject,
                description = description,
                status      = "pending",
                owner       = None,
                blockedBy   = [],
            )
            try:

                with _task_path(task.id).open("x", encoding="utf-8") as handle:

                    json.dump(asdict(task), handle, indent=2)
                #
                return (task)
            #
            except FileExistsError:

                continue
            #
        #
    #
    raise (RuntimeError("Could not allocate a unique task ID"))
#


def _task_depends_on(task_id:str, target_id:str) -> bool:

    """Return whether task_id transitively depends on target_id."""
    pending = [task_id]
    visited = set()
    while (pending):

        current = pending.pop()
        if (current == target_id):

            return (True)
        #
        if (current in visited):

            continue
        #
        visited.add(current)
        pending.extend(load_task(current).blockedBy)
    #
    return (False)
#


def update_task(task_id:str, addBlockedBy:list[str]) -> Task:

    """Add dependency edges after create_task has returned real task IDs."""
    if (not isinstance(addBlockedBy, list)):

        raise (ValueError("addBlockedBy must be a list of task IDs"))
    #

    with task_store_lock():

        task = load_task(task_id)
        if (task.status != "pending"  or  task.owner is not None):

            raise (ValueError(
                f"Task {task_id} dependencies can only be updated while "
                "pending and unowned"
            ))
        #

        dependencies = list(dict.fromkeys(addBlockedBy))
        for dependency in dependencies:

            if (dependency == task_id):

                raise (ValueError("Task cannot depend on itself"))
            #
            if (not _task_path(dependency).is_file()):

                raise (ValueError(f"Dependency not found: {dependency}"))
            #
            if (dependency not in task.blockedBy  and  _task_depends_on(
                dependency, task_id
            )):

                raise (ValueError(
                    f"Dependency cycle detected: {task_id} -> {dependency}"
                ))
            #
        #

        task.blockedBy.extend(
            dependency for dependency in dependencies
            if dependency not in task.blockedBy
        )
        save_task(task)
        return (task)
    #
#




In [ ]:
def save_task(task:Task):

    with task_store_lock():

        path      = _task_path(task.id)
        temporary = path.with_name(
            f".{path.name}.{os.getpid()}.{threading.get_ident()}.tmp"
        )
        try:

            temporary.write_text(
                json.dumps(asdict(task), indent=2), encoding="utf-8"
            )
            os.replace(temporary, path)
        #
        finally:

            temporary.unlink(missing_ok=True)
        #
    #
#


def load_task(task_id:str) -> Task:

    with task_lock:

        data = json.loads(_task_path(task_id).read_text(encoding="utf-8"))
        task = Task(**data)
        if (task.id != task_id):

            raise (ValueError(f"Task file ID does not match {task_id}"))
        #
        if (task.status not in {"pending", "in_progress", "completed"}):

            raise (ValueError(f"Invalid task status: {task.status}"))
        #
        return (task)
    #
#


def list_tasks() -> list[Task]:

    with task_lock:

        if (not TASKS_DIR.exists()):

            return ([])
        #
        if (not TASKS_ROOT.is_relative_to(WORKDIR.resolve())):

            raise (ValueError("Tasks directory escapes workspace"))
        #
        return ([load_task(path.stem)
                for path in sorted(TASKS_DIR.glob("task_*.json"))])
    #
#


def get_task(task_id:str) -> str:

    """Return full task details as JSON."""
    task = load_task(task_id)
    return (json.dumps(asdict(task), indent=2))
#


def can_start(task_id:str) -> bool:

    """Check if all blockedBy dependencies are completed.
    Missing dependencies are treated as blocked."""
    task = load_task(task_id)
    for dep_id in task.blockedBy:

        try:

            dep_path = _task_path(dep_id)
        #
        except ValueError:

            return (False)
        #
        if (not dep_path.exists()):

            return (False)
        #
        if (load_task(dep_id).status != "completed"):

            return (False)
        #
    #
    return (True)
#


def _owner_in_progress(owner:str) -> Task|None:

    return (next((task for task in list_tasks()
                 if task.status == "in_progress"  and  task.owner == owner), None))
#


def _incomplete_dependencies(task:Task) -> list[str]:

    incomplete = []
    for dep_id in task.blockedBy:

        try:

            dep_path = _task_path(dep_id)
        #
        except ValueError:

            incomplete.append(dep_id)
            continue
        #
        if (not dep_path.exists()  or  load_task(dep_id).status != "completed"):

            incomplete.append(dep_id)
        #
    #
    return (incomplete)
#


def claim_task(task_id:str, owner:str="agent") -> str:

    """Atomically claim one task and bind the owner's filesystem cwd."""
    with task_store_lock():

        task = load_task(task_id)
        if (task.status != "pending"):

            return (f"Task {task_id} is {task.status}, cannot claim")
        #
        if (task.owner):

            return (f"Task {task_id} is already owned by {task.owner}")
        #
        assignment = teammate_assignments.get(owner)
        if (assignment):

            return (f"Owner {owner} must finish the current work turn for "
                    f"{assignment['task_id']} before claiming another task")
        #
        current = _owner_in_progress(owner)
        if (current):

            return (f"Owner {owner} must complete {current.id} before "
                    "claiming another task")
        #
        if (not can_start(task_id)):

            return (f"Blocked by: {_incomplete_dependencies(task)}")
        #
        cwd, error = task_worktree_cwd(task)
        if (error):

            return (f"Cannot claim {task_id}: {error}")
        #
        task.owner  = owner
        task.status = "in_progress"
        save_task(task)
        teammate_assignments[owner] = {"task_id": task.id, "cwd": cwd}
        advance_assignment_version(owner)
    #
    print(f"  [claim] {task.subject} -> in_progress (owner: {owner})")
    return (f"Claimed {task.id} ({task.subject})")
#


def complete_task(task_id:str, owner:str="agent") -> str:

    """Complete an assignment only when the caller owns it."""
    with task_store_lock():

        task = load_task(task_id)
        if (task.status != "in_progress"):

            return (f"Task {task_id} is {task.status}, cannot complete")
        #
        if (task.owner != owner):

            return (f"Task {task_id} is owned by {task.owner}, "
                    f"not {owner}; cannot complete")
        #
        gate = globals().get("plan_gates", {}).get(owner, "not_required")
        if (gate in {"required", "pending", "rejected"}):

            return (f"Task {task_id} cannot complete while plan status is {gate}")
        #
        assignment = teammate_assignments.get(owner)
        if (not assignment  or  assignment.get("task_id") != task.id):

            cwd, error = task_worktree_cwd(task)
            if (error):

                return (f"Task {task_id} cannot complete: {error}")
            #
            teammate_assignments[owner] = {"task_id": task.id, "cwd": cwd}
        #
        task.status = "completed"
        save_task(task)
        unblocked = [t.subject for t in list_tasks()
                     if t.status == "pending"  and  t.blockedBy  and  can_start(t.id)]
    #
    print(f"  [complete] {task.subject}")
    msg = f"Completed {task.id} ({task.subject})"
    if (unblocked):

        msg += f"\nUnblocked: {', '.join(unblocked)}"
        print(f"  [unblocked] {', '.join(unblocked)}")
    #
    return (msg)


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Task-bound Worktrees
#

WORKTREES_DIR       = WORKDIR / ".worktrees"
WORKTREES_ROOT      = WORKTREES_DIR.resolve()
VALID_WORKTREE_NAME = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]{0,63}$")


def validate_worktree_name(name:str) -> str|None:

    if (not isinstance(name, str)  or  not VALID_WORKTREE_NAME.fullmatch(name)):

        return ("worktree name must be 1-64 letters, digits, dots, "
                "underscores, or dashes, and start with a letter or digit")
    #
    if (name in {".", ".."}  or  ".." in name):

        return ("worktree name cannot contain '..'")
    #
    return (None)
#


def _worktree_path(name:str) -> Path:

    path = (WORKTREES_DIR / name).resolve()
    if (not WORKTREES_ROOT.is_relative_to(WORKDIR.resolve())
            or  not path.is_relative_to(WORKTREES_ROOT)
            or  path == WORKTREES_ROOT):

        raise (ValueError(f"Worktree path escapes directory: {name!r}"))
    #
    return (path)
#


def _worktree_branch(name:str) -> str:

    return (f"wt/{name}")
#


def _run_git(args:list[str], cwd:Path|None=None) -> tuple[bool, str]:

    """Run Git without shell interpolation and preserve machine output."""
    try:

        result = subprocess.run(
            ["git", *args],
            cwd            = cwd  or  WORKDIR,
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 30,
        )
    #
    except (OSError, subprocess.TimeoutExpired) as exc:

        return (False, f"{type(exc).__name__}: {exc}")
    #
    output = (result.stdout + result.stderr).strip()
    return (result.returncode == 0, output  or  "(no output)")
#


def run_git(args:list[str], cwd:Path|None=None) -> tuple[bool, str]:

    """Run Git and bound only the text returned to the model."""
    ok, output = _run_git(args, cwd)
    return (ok, output[:5000])
#


def _registered_worktrees() -> tuple[dict[Path, dict[str, str]], str|None]:

    ok, output = _run_git(["worktree", "list", "--porcelain"])
    if (not ok):

        return ({}, f"cannot read Git worktree registry: {output}")
    #
    entries : dict[Path, dict[str, str]] = {}
    current : dict[str, str]             = {}
    for line in output.splitlines() + [""]:

        if (not line):

            raw_path = current.get("worktree")
            if (raw_path):

                entries[Path(raw_path).resolve()] = current
            #
            current = {}
            continue
        #
        key, _, value = line.partition(" ")
        current[key]  = value
    #
    return (entries, None)
#


def _registered_worktree(name:str) -> tuple[Path|None, str|None]:

    try:

        path = _worktree_path(name)
    #
    except ValueError as exc:

        return (None, str(exc))
    #
    entries, error = _registered_worktrees()
    if (error):

        return (None, error)
    #
    if (path not in entries):

        return (None, f"worktree '{name}' is not registered with Git")
    #
    if (not path.is_dir()):

        return (None, f"worktree '{name}' is missing at {path}")
    #
    expected_branch = f"refs/heads/{_worktree_branch(name)}"
    if (entries[path].get("branch") != expected_branch):

        return (None, (f"worktree '{name}' is not registered on expected "
                      f"branch '{_worktree_branch(name)}'"))
    #
    return (path, None)
#


def task_worktree_cwd(task:Task) -> tuple[Path, str|None]:

    """Resolve a task cwd, failing closed for broken worktree bindings."""
    if (not task.worktree):

        return (WORKDIR, None)
    #
    path, error = _registered_worktree(task.worktree)
    return ((path  or  WORKDIR), error)
#


def assignment_cwd(owner:str) -> Path:

    with task_lock:

        assignment = teammate_assignments.get(owner)
        task       = _owner_in_progress(owner)
        if (task  and  (not assignment  or  assignment.get("task_id") != task.id)):

            cwd, error = task_worktree_cwd(task)
            if (error):

                raise (ValueError(error))
            #
            assignment                  = {"task_id": task.id, "cwd": cwd}
            teammate_assignments[owner] = assignment
        #
        elif (not assignment):

            return (WORKDIR)
        #
        task = load_task(str(assignment["task_id"]))
        if (task.status not in {"in_progress", "completed"}  or  task.owner != owner):

            raise (ValueError(f"Assignment for {owner} is no longer active"))
        #
        cwd, error = task_worktree_cwd(task)
        if (error):

            raise (ValueError(error))
        #
        if (cwd.resolve() != Path(assignment["cwd"]).resolve()):

            raise (ValueError(f"Assignment cwd changed for task {task.id}"))
        #
        return (cwd)
    #
#




In [ ]:
def release_completed_assignment(owner:str) -> bool:

    """Release a completed cwd lease only at a model turn boundary."""
    with task_lock:

        assignment = teammate_assignments.get(owner)
        if (not assignment):

            return (False)
        #
        task = load_task(str(assignment["task_id"]))
        if (task.status != "completed"  or  task.owner != owner):

            return (False)
        #
        teammate_assignments.pop(owner, None)
        advance_assignment_version(owner)
        if (owner in globals().get("plan_gates", {})):

            globals()["plan_gates"][owner] = "not_required"
        #
        return (True)
    #
#


def release_teammate_assignment(owner:str):

    """Return abandoned teammate work to the task board on thread exit."""
    with task_lock:

        try:

            task = _owner_in_progress(owner)
            if (task):

                task.status = "pending"
                task.owner  = None
                save_task(task)
            #
        #
        finally:

            teammate_assignments.pop(owner, None)
            advance_assignment_version(owner)
            if (owner in globals().get("plan_gates", {})):

                globals()["plan_gates"][owner] = "not_required"
            #
        #
    #
#


def create_worktree(name:str, task_id:str) -> str:

    """Create and bind a dedicated worktree after all inputs validate."""
    error = validate_worktree_name(name)
    if (error):

        return (f"Error: {error}")
    #
    try:

        path      = _worktree_path(name)
        task_path = _task_path(task_id)
    #
    except ValueError as exc:

        return (f"Error: {exc}")
    #
    branch = _worktree_branch(name)

    with task_lock:

        if (not task_path.exists()):

            return (f"Error: Task {task_id} not found")
        #
        task = load_task(task_id)
        if (task.status != "pending"  or  task.owner is not None):

            return (f"Error: Task {task_id} must be pending and unowned")
        #
        if (task.worktree):

            return (f"Error: Task {task_id} already uses worktree '{task.worktree}'")
        #
        if (any(t.worktree == name for t in list_tasks() if t.id != task_id)):

            return (f"Error: Worktree '{name}' is already bound to another task")
        #
        if (path.exists()):

            return (f"Error: Worktree path already exists: {path}")
        #

        ok, root = run_git(["rev-parse", "--show-toplevel"])
        if (not ok  or  Path(root).resolve() != WORKDIR.resolve()):

            return ("Error: Working directory must be the root of a Git repository")
        #
        ok, branch_check = run_git(["check-ref-format", "--branch", branch])
        if (not ok):

            return (f"Error: Invalid worktree branch '{branch}': {branch_check}")
        #
        exists, _ = run_git(["show-ref", "--verify", "--quiet",
                             f"refs/heads/{branch}"])
        if (exists):

            return (f"Error: Branch '{branch}' already exists")
        #
        entries, registry_error = _registered_worktrees()
        if (registry_error):

            return (f"Error: {registry_error}")
        #
        if (path in entries):

            return (f"Error: Worktree path is already registered: {path}")
        #

        WORKTREES_DIR.mkdir(parents=True, exist_ok=True)
        ok, result = run_git(["worktree", "add", "-b", branch,
                              str(path), "HEAD"])
        if (not ok):

            entries, registry_error = _registered_worktrees()
            branch_exists, _        = run_git(
                ["show-ref", "--verify", "--quiet", f"refs/heads/{branch}"]
            )
            artifacts               = []
            if (path.exists()):

                artifacts.append(f"checkout path '{path}'")
            #
            if (registry_error is None  and  path in entries):

                artifacts.append("registered Git worktree")
            #
            if (branch_exists):

                artifacts.append(f"branch '{branch}'")
            #
            if (artifacts):

                return (
                    "Partial operation: git worktree add reported an error "
                    f"after leaving {', '.join(artifacts)}. Task {task_id} "
                    "remains unbound and no Git data was deleted. Run "
                    f"`git worktree list`, inspect '{path}' and '{branch}', "
                    "then keep or remove those artifacts manually after "
                    f"preserving any work. Git error: {result}"
                )
            #
            return (f"Git error: {result}")
        #

        try:

            task.worktree = name
            save_task(task)
        #
        except Exception as exc:

            return (f"Partial success: Worktree '{name}' was created at "
                    f"{path} on branch '{branch}', but task binding failed: "
                    f"{exc}. Git data was retained for manual recovery.")
        #
    #

    print(f"  \033[33m[worktree] created: {name} at {path}\033[0m")
    return (f"Worktree '{name}' created at {path} for task {task_id}")
#


def remove_worktree(name:str, discard_changes:bool=False) -> str:

    """Remove a registered checkout while always retaining its branch."""
    error = validate_worktree_name(name)
    if (error):

        return (f"Error: {error}")
    #

    with task_lock:

        path, error = _registered_worktree(name)
        if (error):

            return (f"Error: {error}")
        #
        bound = [task for task in list_tasks() if task.worktree == name]
        if (not bound):

            return (f"Error: Worktree '{name}' is not bound to a task")
        #
        active = [task for task in bound if task.status != "completed"]
        if (active):

            return (f"Error: Worktree '{name}' is bound to active task "
                    f"{active[0].id}; complete it before removal")
        #
        leased = [owner for owner, assignment in teammate_assignments.items()
                  if Path(assignment["cwd"]).resolve() == path.resolve()]
        if (leased):

            return (f"Error: Worktree '{name}' is still in use by "
                    f"{', '.join(sorted(leased))}; wait for the turn to end")
        #
        ok, status = run_git(
            ["status", "--porcelain", "--ignored"], cwd=path
        )
        if (not ok):

            return (f"Error: Cannot verify worktree '{name}' status: {status}")
        #
        if (status != "(no output)"  and  not discard_changes):

            changed = len([line for line in status.splitlines() if line.strip()])
            return (f"Error: Worktree '{name}' has {changed} uncommitted "
                    "change(s); preserve or discard them manually")
        #

        args = ["worktree", "remove"]
        if (discard_changes):

            args.append("--force")
        #
        args.append(str(path))
        ok, result = run_git(args)
        if (not ok):

            return (f"Git error: {result}")
        #

        try:

            for task in bound:

                task.worktree = None
                save_task(task)
            #
        #
        except Exception as exc:

            return (f"Partial success: Worktree '{name}' was removed and "
                    f"branch '{_worktree_branch(name)}' retained, but task "
                    f"unbinding failed: {exc}. Manual recovery is required.")
        #
    #

    print(f"  [worktree] removed: {name}; branch retained")
    return (f"Worktree '{name}' removed; branch '{_worktree_branch(name)}' retained")


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### System Prompt
#

ENVIRONMENT_PROMPT = (
    ("Windows: the bash tool runs through cmd.exe; use cmd.exe syntax, not Unix "
    "Bash or PowerShell syntax, and prefer dedicated file tools for file operations")
    if (os.name == "nt")
    else ("Unix-like: the bash tool runs the system shell")
)
PROMPT_SECTIONS    = {
    "identity"  : f"You are a coding agent. Act, don't explain. Environment: {ENVIRONMENT_PROMPT}.",
    "tools"     : "Available tools: bash, read_file, write_file, edit_file, glob, "
             "create_task, update_task, list_tasks, get_task, claim_task, "
             "complete_task, "
             "spawn_teammate, list_teammates, send_message, request_shutdown, "
             "request_plan, review_plan, create_worktree.",
    "tasks"     : (
        "Create all task nodes first. Only after create_task returns "
        "runtime-generated IDs, use update_task with those exact IDs to add "
        "dependencies. Only the Lead changes task dependencies."
    ),
    "teams"     : (
        "When parallel work would help, first propose a small team with clear "
        "responsibilities and wait for the user's confirmation. Do not call "
        "spawn_teammate before the user confirms. After confirmation, delegate "
        "independent work by creating a Task for each parallel change. Pass "
        "task_id to spawn_teammate when assigning ready work, then "
        "create a task-bound worktree only when a separate working directory "
        "would prevent conflicting edits. A teammate must complete its current "
        "Task before claiming another. A worktree changes tool default cwd "
        "only; it is not a sandbox. Worktree removal stays with the host or "
        "user. After spawning a teammate, end the current turn instead of "
        "polling its status; the runtime will deliver team events and wake the "
        "Lead. React to those events, and shut teammates down when "
        "coordination is complete."
    ),
    "workspace" : f"Working directory: {WORKDIR}",
}

SYSTEM = "\n\n".join(PROMPT_SECTIONS.values())


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### Base Tools
#

def safe_path(p:str, cwd:Path|None=None) -> Path:

    base = (cwd  or  WORKDIR).resolve()
    path = (base / p).resolve()
    if (not path.is_relative_to(base)):

        raise (ValueError(f"Path escapes workspace: {p}"))
    #
    return (path)
#


def run_bash(command:str, cwd:Path|None=None) -> str:

    try:

        result = subprocess.run(
            command,
            shell          = True,
            cwd            = cwd  or  WORKDIR,
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 120,
        )
        output = (result.stdout + result.stderr).strip()
        output = (output[:50000]) if (output) else ("(no output)")
        if (result.returncode):

            return (f"Error: command exited with status {result.returncode}\n{output}")
        #
        return (output)
    #
    except subprocess.TimeoutExpired:

        return ("Error: Timeout (120s)")
    #
    except OSError as exc:

        return (f"Error: {type(exc).__name__}: {exc}")
    #
#


def run_read(path:str, limit:int|None=None, cwd:Path|None=None) -> str:

    try:

        lines = safe_path(path, cwd).read_text(encoding="utf-8").splitlines()
        if (limit  and  limit < len(lines)):

            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        #
        return ("\n".join(lines))
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#


def run_write(path:str, content:str, cwd:Path|None=None) -> str:

    try:

        fp = safe_path(path, cwd)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content, encoding="utf-8")
        return (f"Wrote {len(content)} bytes to {path}")
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#


def run_edit(path:str, old_text:str, new_text:str, cwd:Path|None=None) -> str:

    try:

        target  = safe_path(path, cwd)
        content = target.read_text(encoding="utf-8")
        count   = content.count(old_text)
        if (count != 1):

            return (f"Error: Expected 1 occurrence, found {count}")
        #
        target.write_text(content.replace(old_text, new_text), encoding="utf-8")
        return (f"Edited {path}")
    #
    except Exception as exc:

        return (f"Error: {exc}")
    #
#


def run_glob(pattern:str, cwd:Path|None=None) -> str:

    try:

        base    = (cwd  or  WORKDIR).resolve()
        matches = [
            str(path.relative_to(base))
            for path in sorted(base.glob(pattern))
            if path.resolve().is_relative_to(base)
        ]
        shown   = matches[:200]
        if (len(matches) > 200):

            shown.append("... (more matches omitted; narrow the pattern)")
        #
        return ("\n".join(shown)  or  "No files found")
    #
    except Exception as exc:

        return (f"Error: {exc}")
    #
#


def _agent_cwd() -> tuple[Path|None, str|None]:

    try:

        return (assignment_cwd("agent"), None)
    #
    except (FileNotFoundError, ValueError) as exc:

        return (None, f"Error: Invalid task assignment: {exc}")
    #
#


def run_agent_bash(command:str) -> str:

    cwd, error = _agent_cwd()
    return (error  or  run_bash(command, cwd))
#


def run_agent_read(path:str, limit:int|None=None) -> str:

    cwd, error = _agent_cwd()
    return (error  or  run_read(path, limit, cwd))
#


def run_agent_write(path:str, content:str) -> str:

    cwd, error = _agent_cwd()
    return (error  or  run_write(path, content, cwd))
#


def run_agent_edit(path:str, old_text:str, new_text:str) -> str:

    cwd, error = _agent_cwd()
    return (error  or  run_edit(path, old_text, new_text, cwd))
#


def run_agent_glob(pattern:str) -> str:

    cwd, error = _agent_cwd()
    return (error  or  run_glob(pattern, cwd))


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Task Tools
#

def run_create_task(subject:str, description:str="") -> str:

    task = create_task(subject, description)
    print(f"  \033[34m[create] {task.subject}\033[0m")
    return (f"Created {task.id}: {task.subject}")
#


def run_update_task(task_id:str, addBlockedBy:list[str]) -> str:

    try:

        task = update_task(task_id, addBlockedBy)
    #
    except ValueError as exc:

        return (f"Error: {exc}")
    #
    except FileNotFoundError:

        return (f"Error: Task {task_id} not found")
    #
    dependencies = ", ".join(task.blockedBy)  or  "(none)"
    print(f"  \033[34m[update] {task.subject} blockedBy: {dependencies}\033[0m")
    return (f"Updated {task.id} blockedBy: {dependencies}")
#


def run_list_tasks() -> str:

    tasks = list_tasks()
    if (not tasks):

        return ("No tasks. Use create_task to add some.")
    #
    lines = []
    for t in tasks:

        icon     = {
            "pending"     : "[ ]",
            "in_progress" : "[~]",
            "completed"   : "[x]",
        }.get(t.status, "[?]")
        deps     = (f" (blockedBy: {', '.join(t.blockedBy)})") if (t.blockedBy) else ("")
        owner    = (f" [{t.owner}]") if (t.owner) else ("")
        worktree = (f" (worktree: {t.worktree})") if (t.worktree) else ("")
        lines.append(f"  {icon} {t.id}: {t.subject} "
                     f"[{t.status}]{owner}{deps}{worktree}")
    #
    return ("\n".join(lines))
#


def run_get_task(task_id:str) -> str:

    try:

        return (get_task(task_id))
    #
    except ValueError as exc:

        return (f"Error: {exc}")
    #
    except FileNotFoundError:

        return (f"Error: Task {task_id} not found")
    #
#


def run_claim_task(task_id:str) -> str:

    try:

        return (claim_task(task_id, owner="agent"))
    #
    except ValueError as exc:

        return (f"Error: {exc}")
    #
    except FileNotFoundError:

        return (f"Error: Task {task_id} not found")
    #
#


def run_complete_task(task_id:str) -> str:

    try:

        return (complete_task(task_id, owner="agent"))
    #
    except ValueError as exc:

        return (f"Error: {exc}")
    #
    except FileNotFoundError:

        return (f"Error: Task {task_id} not found")


In [ ]:
    #
#


#
###################################################################################################
###################################################################################################
################################################################################################### MessageBus and Team Protocols
#


MAILBOX_DIR             = WORKDIR / ".mailboxes"
MAILBOX_ROOT            = MAILBOX_DIR.resolve()
VALID_AGENT_NAME        = re.compile(r"^[A-Za-z0-9_-]{1,64}$")
RESERVED_TEAMMATE_NAMES = {"lead", "agent"}


def is_valid_agent_name(name:str) -> bool:

    return (bool(VALID_AGENT_NAME.fullmatch(name)))
#


class MessageBus:

    """Thread-safe file mailboxes with destructive reads."""

    #
    #################################################################  Inits
    #
    def __init__(self):

        self._lock    = threading.RLock()
        self._changed = threading.Condition(self._lock)
    #

    #
    #################################################################  Internals
    #
    def _path(self, agent:str) -> Path:

        if (not is_valid_agent_name(agent)):

            raise (ValueError(f"Invalid mailbox recipient: {agent!r}"))
        #
        path = (MAILBOX_DIR / f"{agent}.jsonl").resolve()
        if (not path.is_relative_to(MAILBOX_ROOT)):

            raise (ValueError(f"Mailbox path escapes directory: {agent!r}"))
        #
        return (path)
    #

    #
    #################################################################  Internals
    #
    def _read_unlocked(self, agent:str) -> list[dict]:

        inbox = self._path(agent)
        if (not inbox.exists()):

            return ([])
        #
        msgs = [json.loads(line) for line in inbox.read_text(encoding="utf-8").splitlines()
                if line.strip()]
        inbox.unlink()
        return (msgs)
    #

    #
    #################################################################  Public API
    #
    def send(self, from_agent:str, to_agent:str, content:str, msg_type:str="message", metadata:dict|None=None):

        msg = {
            "from"     : from_agent,
            "to"       : to_agent,
            "content"  : content,
            "type"     : msg_type,
            "ts"       : time.time(),
            "metadata" : metadata  or  {},
        }
        with self._changed:

            MAILBOX_DIR.mkdir(parents=True, exist_ok=True)
            with self._path(to_agent).open("a", encoding="utf-8") as handle:

                handle.write(json.dumps(msg, ensure_ascii=True) + "\n")
            #
            self._changed.notify_all()
        #
        print(f"  [bus] {from_agent} -> {to_agent}: "
              f"({msg_type}) {content[:50]}")
    #

    #
    #################################################################  Public API
    #
    def read_inbox(self, agent:str) -> list[dict]:

        with self._lock:

            return (self._read_unlocked(agent))
        #
    #

    def peek(self, agent:str) -> bool:

        with self._lock:

            inbox = self._path(agent)
            return (inbox.exists()  and  inbox.stat().st_size > 0)
        #
    #

    def wait_for_messages(self, agent:str, timeout:float|None=None) -> list[dict]:

        """Block until the agent has messages or timeout expires."""
        deadline = (None) if (timeout is None) else (time.monotonic() + timeout)
        with self._changed:

            while (not self.peek(agent)):

                remaining = ((None) if (deadline is None)
                             else (deadline - time.monotonic()))
                if (remaining is not None  and  remaining <= 0):

                    return ([])
                #
                self._changed.wait(remaining)
            #
            return (self._read_unlocked(agent))
        #
    #
#


BUS = MessageBus()

# working | waiting_approval | idle | stopping
active_teammates: dict[str, str] = {}
plan_gates: dict[str, str] = {}
plan_request_ids: dict[str, str] = {}
team_lock = threading.RLock()


@dataclass
class ProtocolState:

    request_id   : str
    type         : str
    sender       : str
    target       : str
    status       : str
    payload      : str
    work_version : int|None = None
    task_id      : str|None = None
    created_at   : float    = field(default_factory=time.time)
#


pending_requests: dict[str, ProtocolState] = {}


def new_request_id() -> str:

    while (True):

        request_id = f"req_{random.randint(0, 999999):06d}"
        if (request_id not in pending_requests):

            return (request_id)
        #
    #
#


def match_response(response_type:str, request_id:str, approve:bool, from_agent:str, to_agent:str) -> bool:

    """Match one protocol response to one pending request."""
    with team_lock:

        state = pending_requests.get(request_id)
        if (not state):

            print(f"  [protocol] unknown request_id: {request_id}")
            return (False)
        #
        expected = {
            "shutdown"      : "shutdown_response",
            "plan_approval" : "plan_approval_response",
        }[state.type]
        if (response_type != expected):

            print(f"  [protocol] expected {expected}, got {response_type}")
            return (False)
        #
        if (from_agent != state.target  or  to_agent != state.sender):

            print(f"  [protocol] {request_id} responder mismatch")
            return (False)
        #
        if (state.status != "pending"):

            print(f"  [protocol] {request_id} already {state.status}")
            return (False)
        #
        state.status = ("approved") if (approve) else ("rejected")
    #
    print(f"  [protocol] {request_id} -> {state.status}")
    return (True)
#




In [ ]:
def consume_lead_inbox() -> list[dict]:

    """Consume Lead events and update protocol state before model delivery."""
    msgs = BUS.read_inbox("lead")
    for msg in msgs:

        metadata   = msg.get("metadata", {})
        request_id = metadata.get("request_id", "")
        if (request_id  and  msg.get("type", "").endswith("_response")):

            match_response(msg["type"], request_id,
                           metadata.get("approve", False),
                           msg.get("from", ""), msg.get("to", ""))
        #
    #
    return (msgs)
#


def format_team_events(msgs:list[dict]) -> str:

    lines = []
    for msg in msgs:

        metadata   = msg.get("metadata", {})
        request_id = metadata.get("request_id")
        suffix     = (f" request_id={request_id}") if (request_id) else ("")
        lines.append(
            f"[{msg['type']}{suffix}] {msg['from']}: {msg['content']}"
        )
    #
    return ("[Team events]\n" + "\n".join(lines))
#


def _last_assistant_text(content) -> str:

    for block in content:

        if (getattr(block, "type", None) == "text"):

            return (block.text.strip())
        #
        if (isinstance(block, dict)  and  block.get("type") == "text"):

            return (str(block.get("text", "")).strip())
        #
    #
    return ("")
#


def current_work_identity(owner:str) -> tuple[int, str|None]:

    with task_lock:

        assignment = teammate_assignments.get(owner)
        task_id    = (str(assignment["task_id"])) if (assignment) else (None)
        return (assignment_versions.get(owner, 0), task_id)
    #
#


def _teammate_submit_plan(from_name:str, plan:str) -> str:

    with task_lock:

        assignment   = teammate_assignments.get(from_name)
        task_id      = (str(assignment["task_id"])) if (assignment) else (None)
        work_version = assignment_versions.get(from_name, 0)
        with team_lock:

            if (plan_gates.get(from_name) == "pending"):

                return ("A plan is already waiting for review.")
            #
            request_id                   = new_request_id()
            pending_requests[request_id] = ProtocolState(
                request_id   = request_id,
                type         = "plan_approval",
                sender       = from_name,
                target       = "lead",
                status       = "pending",
                payload      = plan,
                work_version = work_version,
                task_id      = task_id,
            )
            plan_gates[from_name]        = "pending"
            plan_request_ids[from_name]  = request_id
            active_teammates[from_name]  = "waiting_approval"
        #
    #
    BUS.send(from_name, "lead", plan, "plan_approval_request",
             {"request_id": request_id})
    return (f"Plan submitted ({request_id}). Wait for Lead's decision.")
#


def _run_teammate_tool(name:str, block, handlers:dict) -> str:

    gate = plan_gates.get(name, "not_required")
    if (block.name in {"bash", "write_file", "edit_file"}):

        if (gate != "approved"):

            if (gate != "not_required"):

                return (f"Blocked: plan status is {gate}. Submit or revise the "
                        "plan and wait for approval before changing the workspace.")
            #
        #
        blocked = check_permission(block, prompt_user=False)
        if (blocked):

            return (blocked)
        #
    #
    handler = handlers.get(block.name)
    if (not handler):

        return (f"Unknown tool: {block.name}")
    #
    trigger_hooks("PreToolUse", block, skip_permission=True)
    try:

        output = str(handler(**block.input))
    #
    except Exception as exc:

        output = f"Error: {type(exc).__name__}: {exc}"
    #
    trigger_hooks("PostToolUse", block, output)
    return (output)
#


def apply_plan_response(name:str, msg:dict) -> tuple[bool, str]:

    """Apply only the Lead response for this teammate's current plan."""
    metadata              = msg.get("metadata", {})
    request_id            = metadata.get("request_id", "")
    work_version, task_id = current_work_identity(name)
    with team_lock:

        state       = pending_requests.get(request_id)
        expected_id = plan_request_ids.get(name)
        valid       = (
            msg.get("from") == "lead"
            and  msg.get("to") == name
            and  request_id == expected_id
            and  state is not None
            and  state.type == "plan_approval"
            and  state.sender == name
            and  state.target == "lead"
            and  state.work_version == work_version
            and  state.task_id == task_id
            and  state.status in {"approved", "rejected"}
            and  metadata.get("approve", False)
            == (state.status == "approved")
        )
        if (not valid):

            return (False, "[Ignored plan response: request mismatch]")
        #
        plan_gates[name]       = state.status
        active_teammates[name] = "working"
        plan_request_ids.pop(name, None)
        outcome = state.status
    #
    return (True, f"[Plan {outcome}] {msg['content']}")
#


def apply_shutdown_request(name:str, msg:dict) -> tuple[bool, str]:

    """Accept only a pending shutdown request sent by Lead to this teammate."""
    request_id = msg.get("metadata", {}).get("request_id", "")
    with team_lock:

        state = pending_requests.get(request_id)
        valid = (
            msg.get("from") == "lead"
            and  msg.get("to") == name
            and  state is not None
            and  state.type == "shutdown"
            and  state.sender == "lead"
            and  state.target == name
            and  state.status == "pending"
            and  active_teammates.get(name) != "stopping"
        )
        if (not valid):

            return (False, "[Ignored shutdown request: request mismatch]")
        #
        active_teammates[name] = "stopping"
    #
    return (True, request_id)
#


def _teammate_send_message(from_name:str, to:str, content:str) -> str:

    with team_lock:

        if (to != "lead"  and  to not in active_teammates):

            return (f"Agent '{to}' is not active")
        #
    #
    BUS.send(from_name, to, content)
    return (f"Sent to {to}")


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Idle Task Discovery
#

IDLE_SCAN_INTERVAL = 2.0


def scan_unclaimed_tasks() -> list[Task]:

    """Return ready tasks whose optional worktree binding is usable."""
    with task_lock:

        ready = []
        for task in list_tasks():

            if (task.status != "pending"  or  task.owner is not None
                    or  not can_start(task.id)):

                continue
            #
            _, error = task_worktree_cwd(task)
            if (not error):

                ready.append(task)
            #
        #
        return (ready)
    #
#


def claim_next_task(name:str) -> Task|None:

    """Claim the first still-available task, never a second assignment."""
    with task_lock:

        if (teammate_assignments.get(name)  or  _owner_in_progress(name)):

            return (None)
        #
    #
    for task in scan_unclaimed_tasks():

        result = claim_task(task.id, owner=name)
        if (result.startswith("Claimed ")):

            return (load_task(task.id))
        #
    #
    return (None)


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Teammate Runtime
#


class TeammateRuntime:

    """One persistent teammate with separate messages and WORK/IDLE phases."""

    #
    #################################################################  Inits
    #
    def __init__(self, name:str, role:str, prompt:str, task_id:str|None, require_plan:bool):

        self.name     = name
        self.system   = (
            f"You are '{name}', a {role}. Use tools to complete the assigned "
            "Task, then call complete_task and report a concise result. "
            "If the first user message contains [Assigned task], that Task is "
            "already claimed; do not call claim_task for it again. "
            "When asked for a plan, call submit_plan and wait for approval "
            "before bash or file changes. File and shell tools use the Task's "
            "working directory; that directory is not a sandbox. The runtime "
            "delivers your final text to Lead. Use send_message only for "
            "intermediate coordination, and address the coordinator as 'lead'."
        )
        self.messages = [{"role": "user", "content": prompt}]
        if (task_id):

            task = load_task(task_id)
            cwd  = assignment_cwd(name)
            self.messages[0]["content"] += (
                f"\n\n[Assigned task {task.id}] {task.subject}\n"
                f"{task.description}\nWork directory: {cwd}"
            )
        #
        if (require_plan):

            self.messages[0]["content"] += (
                "\n\n[Plan required] Submit a plan and wait for Lead approval "
                "before changing files or using bash."
            )
        #
        self.handlers = {
            "bash"          : self.bash,
            "read_file"     : self.read,
            "write_file"    : self.write,
            "edit_file"     : self.edit,
            "glob"          : self.glob,
            "send_message"  : lambda to, content: _teammate_send_message(
                name, to, content),
            "submit_plan"   : lambda plan: _teammate_submit_plan(name, plan),
            "list_tasks"    : run_list_tasks,
            "claim_task"    : self.claim,
            "complete_task" : self.complete,
        }
    #

    #
    #################################################################  Public API
    #
    def current_cwd(self) -> tuple[Path|None, str|None]:

        if (self.name not in teammate_assignments):

            return (None, "Error: Claim a Task before using workspace tools.")
        #
        try:

            return (assignment_cwd(self.name), None)
        #
        except (FileNotFoundError, ValueError) as exc:

            return (None, f"Error: Invalid task assignment: {exc}")
        #
    #

    #
    #################################################################  Public API
    #
    def bash(self, command:str) -> str:

        cwd, error = self.current_cwd()
        return (error  or  run_bash(command, cwd=cwd))
    #

    def read(self, path:str, limit:int|None=None) -> str:

        cwd, error = self.current_cwd()
        return (error  or  run_read(path, limit=limit, cwd=cwd))
    #

    def write(self, path:str, content:str) -> str:

        cwd, error = self.current_cwd()
        return (error  or  run_write(path, content, cwd=cwd))
    #

    def edit(self, path:str, old_text:str, new_text:str) -> str:

        cwd, error = self.current_cwd()
        return (error  or  run_edit(path, old_text, new_text, cwd=cwd))
    #

    def glob(self, pattern:str) -> str:

        cwd, error = self.current_cwd()
        return (error  or  run_glob(pattern, cwd=cwd))
    #

    def claim(self, task_id:str) -> str:

        try:

            return (claim_task(task_id, owner=self.name))
        #
        except ValueError as exc:

            return (f"Error: {exc}")
        #
        except FileNotFoundError:

            return (f"Error: Task {task_id} not found")
        #
    #

    def complete(self, task_id:str) -> str:

        try:

            return (complete_task(task_id, owner=self.name))
        #
        except ValueError as exc:

            return (f"Error: {exc}")
        #
        except FileNotFoundError:

            return (f"Error: Task {task_id} not found")
        #
    #

    def handle_inbox(self, inbox:list[dict]) -> bool:

        """Append work messages and return True for a valid shutdown."""
        work_messages = []
        for msg in inbox:

            msg_type = msg.get("type", "message")
            if (msg_type == "shutdown_request"):

                accepted, notice = apply_shutdown_request(self.name, msg)
                if (not accepted):

                    work_messages.append(notice)
                    continue
                #
                BUS.send(self.name, "lead", "Shutdown acknowledged.",
                         "shutdown_response",
                         {"request_id": notice, "approve": True})
                return (True)
            #
            if (msg_type == "plan_approval_response"):

                _, notice = apply_plan_response(self.name, msg)
                work_messages.append(notice)
                continue
            #
            if (msg_type == "plan_request"):

                work_messages.append(f"[Plan required] {msg['content']}")
                continue
            #
            work_messages.append(
                f"[Message from {msg['from']}] {msg['content']}"
            )
        #
        if (work_messages):

            self.messages.append({
                "role"    : "user",
                "content" : "\n".join(work_messages),
            })
        #
        return (False)
    #

    def work(self) -> str:

        """Run one model turn. Return continue, idle, or stop."""
        if (self.handle_inbox(BUS.read_inbox(self.name))):

            return ("stop")
        #
        with team_lock:

            active_teammates[self.name] = "working"
        #
        try:

            response = client.messages.create(
                model      = MODEL,
                system     = self.system,
                messages   = self.messages,
                tools      = TEAMMATE_TOOLS,
                max_tokens = 8000,
            )
        #
        except Exception as exc:

            BUS.send(self.name, "lead",
                     f"{type(exc).__name__}: {exc}", "error")
            return ("stop")
        #

        self.messages.append({
            "role"    : "assistant",
            "content" : response.content,
        })
        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if (tool_calls):

            results = []
            for block in tool_calls:

                output = _run_teammate_tool(
                    self.name, block, self.handlers
                )
                results.append({
                    "type"        : "tool_result",
                    "tool_use_id" : block.id,
                    "content"     : output,
                })
            #
            self.messages.append({"role": "user", "content": results})
            return ("continue")
        #

        summary = _last_assistant_text(response.content)
        gate    = plan_gates.get(self.name, "not_required")
        if (gate != "pending"  and  summary):

            BUS.send(self.name, "lead", summary, "result")
        #
        if (gate == "pending"):

            with team_lock:

                active_teammates[self.name] = "waiting_approval"
            #
        #
        else:

            release_completed_assignment(self.name)
            with team_lock:

                active_teammates[self.name] = "idle"
            #
            BUS.send(self.name, "lead", "Waiting for more work.",
                     "idle_notification")
        #
        return ("idle")
    #

    def wait_for_work(self) -> bool:

        """Wait for a message or atomically claim the next ready Task."""
        while (True):

            inbox = BUS.wait_for_messages(self.name, IDLE_SCAN_INTERVAL)
            if (inbox):

                before = len(self.messages)
                if (self.handle_inbox(inbox)):

                    return (False)
                #
                if (len(self.messages) > before):

                    return (True)
                #
                continue
            #

            task = claim_next_task(self.name)
            if (not task):

                continue
            #
            cwd = assignment_cwd(self.name)
            self.messages.append({
                "role"    : "user",
                "content" : (
                    f"[Auto-claimed task {task.id}] {task.subject}\n"
                    f"{task.description}\nWork directory: {cwd}"
                ),
            })
            print(f"  [idle] {self.name} claimed {task.id}: {task.subject}")
            return (True)
        #
    #

    def run(self):

        try:

            state = "continue"
            while (state != "stop"):

                if (state == "idle"  and  not self.wait_for_work()):

                    break
                #
                state = self.work()
            #
        #
        except Exception as exc:

            try:

                BUS.send(self.name, "lead",
                         f"{type(exc).__name__}: {exc}", "error")
            #
            except Exception:

                pass
            #
        #
        finally:

            try:

                release_teammate_assignment(self.name)
            #
            except Exception as exc:

                try:

                    BUS.send(
                        self.name, "lead",
                        f"Assignment cleanup failed: {type(exc).__name__}: {exc}",
                        "error",
                    )
                #
                except Exception:

                    pass
                #
            #
            with team_lock:

                active_teammates.pop(self.name, None)
                plan_gates.pop(self.name, None)
                plan_request_ids.pop(self.name, None)
                teammate_threads.pop(self.name, None)
            #
            print(f"  [teammate] {self.name} finished")
        #
    #
#


teammate_threads: dict[str, threading.Thread] = {}




In [ ]:
def spawn_teammate_thread(name:str, role:str, prompt:str, task_id:str|None=None, require_plan:bool=False) -> str:

    """Claim an initial Task, then start one persistent teammate."""
    if (not is_valid_agent_name(name)):

        return ("Invalid teammate name: use 1-64 letters, digits, "
                "underscores, or dashes")
    #
    if (name.lower() in RESERVED_TEAMMATE_NAMES):

        return (f"Invalid teammate name: '{name}' is reserved by the runtime")
    #
    with team_lock:

        if (any(existing.casefold() == name.casefold()
               for existing in active_teammates)):

            return (f"Teammate '{name}' already exists")
        #
        active_teammates[name]    = "working"
        plan_gates[name]          = ("required") if (require_plan) else ("not_required")
        assignment_versions[name] = 0
    #

    if (task_id):

        try:

            claimed = claim_task(task_id, owner=name)
        #
        except (FileNotFoundError, ValueError) as exc:

            claimed = f"Error: {exc}"
        #
        if (not claimed.startswith("Claimed ")):

            with team_lock:

                active_teammates.pop(name, None)
                plan_gates.pop(name, None)
                assignment_versions.pop(name, None)
            #
            return (f"Cannot spawn teammate '{name}': {claimed}")
        #
    #

    runtime = TeammateRuntime(name, role, prompt, task_id, require_plan)
    thread  = threading.Thread(target=runtime.run, daemon=True)
    with team_lock:

        teammate_threads[name] = thread
    #
    thread.start()
    print(f"  [teammate] {name} spawned as {role}")
    assigned = (f" for {task_id}") if (task_id) else (" without an initial Task")
    return (
        f"Teammate '{name}' spawned as {role}{assigned}. "
        "End this turn; the runtime will deliver its events."
    )


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Lead Team Tools
#

def run_spawn_teammate(name:str, role:str, prompt:str, task_id:str|None=None, require_plan:bool=False) -> str:

    return (spawn_teammate_thread(name, role, prompt, task_id, require_plan))
#


def run_list_teammates() -> str:

    with team_lock:

        if (not active_teammates):

            return ("No active teammates.")
        #
        return ("\n".join(
            f"{name}: {status}"
            for name, status in sorted(active_teammates.items())
        ))
    #
#


def run_send_message(to:str, content:str) -> str:

    if (to not in active_teammates):

        return (f"Teammate '{to}' is not active")
    #
    BUS.send("lead", to, content)
    return (f"Sent to {to}")
#


def run_request_shutdown(teammate:str) -> str:

    if (teammate not in active_teammates):

        return (f"Teammate '{teammate}' is not active")
    #
    with team_lock:

        request_id                   = new_request_id()
        pending_requests[request_id] = ProtocolState(
            request_id = request_id,
            type       = "shutdown",
            sender     = "lead",
            target     = teammate,
            status     = "pending",
            payload    = "",
        )
    #
    BUS.send("lead", teammate, "Finish the current step and shut down.",
             "shutdown_request", {"request_id": request_id})
    return (f"Shutdown requested from {teammate} ({request_id})")
#


def run_request_plan(teammate:str, task:str) -> str:

    if (teammate not in active_teammates):

        return (f"Teammate '{teammate}' is not active")
    #
    with team_lock:

        plan_gates[teammate] = "required"
    #
    BUS.send("lead", teammate, task, "plan_request")
    return (f"Plan requested from {teammate}")
#


def run_review_plan(request_id:str, approve:bool, feedback:str="") -> str:

    state = pending_requests.get(request_id)
    if (not state):

        return (f"Request {request_id} not found")
    #
    work_version, task_id = current_work_identity(state.sender)
    with team_lock:

        state = pending_requests.get(request_id)
        if (not state):

            return (f"Request {request_id} not found")
        #
        if (state.type != "plan_approval"):

            return (f"Request {request_id} is not a plan")
        #
        if (state.status != "pending"):

            return (f"Request {request_id} already {state.status}")
        #
        if (state.work_version != work_version  or  state.task_id != task_id):

            return (f"Request {request_id} belongs to an earlier assignment")
        #
        if (plan_request_ids.get(state.sender) != request_id):

            return (f"Request {request_id} is not the current plan")
        #
        state.status = ("approved") if (approve) else ("rejected")
    #
    content = feedback  or  (("Plan approved.") if (approve)
                           else ("Revise the plan and submit it again."))
    BUS.send("lead", state.sender, content, "plan_approval_response",
             {"request_id": request_id, "approve": approve})
    return (f"Plan {state.status} ({request_id})")
#


def run_create_worktree(name:str, task_id:str) -> str:

    return (create_worktree(name, task_id))


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Tool Definitions
#

BASE_TOOLS = [
    {
        "name"         : "bash",
        "description"  : "Run a shell command.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"command": {"type": "string"}},
            "required"   : ["command"],
        },
    },
    {
        "name"         : "read_file",
        "description"  : "Read file contents.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"  : {"type": "string"},
                "limit" : {"type": "integer"},
            },
            "required"   : ["path"],
        },
    },
    {
        "name"         : "write_file",
        "description"  : "Write content to a file.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"    : {"type": "string"},
                "content" : {"type": "string"},
            },
            "required"   : ["path", "content"],
        },
    },
    {
        "name"         : "edit_file",
        "description"  : "Replace exact text once.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"     : {"type": "string"},
                "old_text" : {"type": "string"},
                "new_text" : {"type": "string"},
            },
            "required"   : ["path", "old_text", "new_text"],
        },
    },
    {
        "name"         : "glob",
        "description"  : "Find files by glob pattern; ** matches recursively.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"pattern": {"type": "string"}},
            "required"   : ["pattern"],
        },
    },
]

TASK_TOOLS = [
    {
        "name"         : "create_task",
        "description"  : "Create a task and return its runtime-generated ID.",
        "input_schema" : {
            "type"                 : "object",
            "properties"           : {
                "subject"     : {"type": "string"},
                "description" : {"type": "string"},
            },
            "required"             : ["subject"],
            "additionalProperties" : False,
        },
    },
    {
        "name"         : "update_task",
        "description"  : "Add dependencies using IDs returned by create_task.",
        "input_schema" : {
            "type"                 : "object",
            "properties"           : {
                "task_id"      : {
                    "type"    : "string",
                    "pattern" : "^task_[0-9a-f]{8}$",
                },
                "addBlockedBy" : {
                    "type"     : "array",
                    "items"    : {
                        "type"    : "string",
                        "pattern" : "^task_[0-9a-f]{8}$",
                    },
                    "minItems" : 1,
                },
            },
            "required"             : ["task_id", "addBlockedBy"],
            "additionalProperties" : False,
        },
    },
    {
        "name"         : "list_tasks",
        "description"  : "List shared tasks.",
        "input_schema" : {"type": "object", "properties": {}},
    },
    {
        "name"         : "get_task",
        "description"  : "Get one task by ID.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"task_id": {"type": "string"}},
            "required"   : ["task_id"],
        },
    },
    {
        "name"         : "claim_task",
        "description"  : "Claim a ready task.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"task_id": {"type": "string"}},
            "required"   : ["task_id"],
        },
    },
    {
        "name"         : "complete_task",
        "description"  : "Complete an owned task.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"task_id": {"type": "string"}},
            "required"   : ["task_id"],
        },
    },
]

TEAMMATE_TOOLS = [
    *BASE_TOOLS,
    {
        "name"         : "send_message",
        "description"  : "Send an intermediate message to 'lead' or an active teammate.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "to"      : {"type": "string"},
                "content" : {"type": "string"},
            },
            "required"   : ["to", "content"],
        },
    },
    {
        "name"         : "submit_plan",
        "description"  : "Submit a work plan for Lead approval.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"plan": {"type": "string"}},
            "required"   : ["plan"],
        },
    },
    next(tool for tool in TASK_TOOLS if tool["name"] == "list_tasks"),
    next(tool for tool in TASK_TOOLS if tool["name"] == "claim_task"),
    next(tool for tool in TASK_TOOLS if tool["name"] == "complete_task"),
]

TEAM_TOOLS = [
    {
        "name"         : "spawn_teammate",
        "description"  : "Spawn a persistent teammate.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "name"         : {
                    "type"    : "string",
                    "pattern" : "^[A-Za-z0-9_-]{1,64}$",
                },
                "role"         : {"type": "string"},
                "prompt"       : {"type": "string"},
                "task_id"      : {
                    "type"    : "string",
                    "pattern" : "^task_[0-9a-f]{8}$",
                },
                "require_plan" : {"type": "boolean"},
            },
            "required"   : ["name", "role", "prompt"],
        },
    },
    {
        "name"         : "list_teammates",
        "description"  : "List active teammates.",
        "input_schema" : {"type": "object", "properties": {}},
    },
    {
        "name"         : "send_message",
        "description"  : "Message a teammate.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "to"      : {"type": "string"},
                "content" : {"type": "string"},
            },
            "required"   : ["to", "content"],
        },
    },
    {
        "name"         : "request_shutdown",
        "description"  : "Ask a teammate to shut down.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"teammate": {"type": "string"}},
            "required"   : ["teammate"],
        },
    },
    {
        "name"         : "request_plan",
        "description"  : "Require a teammate plan before workspace changes.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "teammate" : {"type": "string"},
                "task"     : {"type": "string"},
            },
            "required"   : ["teammate", "task"],
        },
    },
    {
        "name"         : "review_plan",
        "description"  : "Approve or reject a plan.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "request_id" : {"type": "string"},
                "approve"    : {"type": "boolean"},
                "feedback"   : {"type": "string"},
            },
            "required"   : ["request_id", "approve"],
        },
    },
    {
        "name"         : "create_worktree",
        "description"  : "Create and bind a task worktree.",
        "input_schema" : {
            "type"                 : "object",
            "properties"           : {
                "name"    : {
                    "type"      : "string",
                    "pattern"   : "^(?!.*\\.\\.)[A-Za-z0-9][A-Za-z0-9._-]{0,63}$",
                    "maxLength" : 64,
                },
                "task_id" : {"type": "string"},
            },
            "required"             : ["name", "task_id"],
            "additionalProperties" : False,
        },
    },
]

TOOLS = [*BASE_TOOLS, *TASK_TOOLS, *TEAM_TOOLS]

TOOL_HANDLERS = {
    "bash"             : run_agent_bash,
    "read_file"        : run_agent_read,
    "write_file"       : run_agent_write,
    "edit_file"        : run_agent_edit,
    "glob"             : run_agent_glob,
    "create_task"      : run_create_task,
    "update_task"      : run_update_task,
    "list_tasks"       : run_list_tasks,
    "get_task"         : run_get_task,
    "claim_task"       : run_claim_task,
    "complete_task"    : run_complete_task,
    "spawn_teammate"   : run_spawn_teammate,
    "list_teammates"   : run_list_teammates,
    "send_message"     : run_send_message,
    "request_shutdown" : run_request_shutdown,
    "request_plan"     : run_request_plan,
    "review_plan"      : run_review_plan,
    "create_worktree"  : run_create_worktree,
}


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### Hooks and Permission Checks
#

HOOKS                    = {"UserPromptSubmit": [], "PreToolUse": [], "PostToolUse": [], "Stop": []}
DENY_LIST                = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "dd if="]
DESTRUCTIVE_COMMAND_WORD = re.compile(
    r"(?i)(?:^|[;&|()\n])\s*(?:rm|del)(?=\s|$|[;&|()])"
)
DESTRUCTIVE              = ["rm ", "> /etc/", "chmod 777"]


def contains_destructive_command(command:str) -> bool:

    return (bool(DESTRUCTIVE_COMMAND_WORD.search(command)))
#


def register_hook(event:str, callback):

    HOOKS[event].append(callback)
#


def trigger_hooks(event:str, *args, skip_permission:bool=False):

    for callback in HOOKS[event]:

        if (skip_permission  and  callback is permission_hook):

            continue
        #
        result = callback(*args)
        if (result is not None):

            return (result)
        #
    #
    return (None)
#


def check_permission(block, prompt_user:bool=True) -> str|None:

    if (block.name == "bash"):

        command = block.input.get("command", "")
        for pattern in DENY_LIST:

            if (pattern in command):

                return (f"Permission denied by deny list: {pattern}")
            #
        #
        if (contains_destructive_command(command)  or  any(
            keyword in command for keyword in DESTRUCTIVE
        )):

            if (not prompt_user):

                return ("Permission required: ask Lead to run this command.")
            #
            print(f"\n[permission] {block.name}({block.input})")
            if (input("Allow? [y/N] ").strip().lower() not in {"y", "yes"}):

                return ("Permission denied by user")
            #
        #
    #

    if (block.name in {"read_file", "write_file", "edit_file"}):

        raw_path = block.input.get("path", "")
        if (not (WORKDIR / raw_path).resolve().is_relative_to(WORKDIR.resolve())):

            if (not prompt_user):

                return ("Permission required: path is outside the workspace.")
            #
            print(f"\n[permission] {block.name}({block.input})")
            if (input("Allow? [y/N] ").strip().lower() not in {"y", "yes"}):

                return ("Permission denied by user")
            #
        #
    #
    return (None)
#


def permission_hook(block):

    return (check_permission(block, prompt_user=True))
#


def log_hook(block):

    preview = str(list(block.input.values())[:2])[:60]
    print(f"[hook] {block.name}({preview})")
    return (None)
#


def large_output_hook(block, output):

    if (len(str(output)) > 100000):

        print(f"[hook] Large output from {block.name}: {len(str(output))} chars")
    #
    return (None)
#


def context_hook(query:str):

    print(f"[hook] UserPromptSubmit: working in {WORKDIR}")
    return (None)
#


def summary_hook(messages:list):

    tool_count = sum(
        1
        for message in messages
        for block in (
            (message.get("content"))
            if (isinstance(message.get("content"), list))
            else ([])
        )
        if isinstance(block, dict)  and  block.get("type") == "tool_result"
    )
    print(f"[hook] Stop: session used {tool_count} tool calls")
    return (None)
#


register_hook("UserPromptSubmit", context_hook)
register_hook("PreToolUse", permission_hook)
register_hook("PreToolUse", log_hook)
register_hook("PostToolUse", large_output_hook)
register_hook("Stop", summary_hook)


def execute_tool(block) -> str:

    blocked = trigger_hooks("PreToolUse", block)
    if (blocked):

        return (str(blocked))
    #
    handler = TOOL_HANDLERS.get(block.name)
    if (not handler):

        return (f"Unknown tool: {block.name}")
    #
    try:

        output = str(handler(**block.input))
    #
    except Exception as exc:

        output = f"Error: {type(exc).__name__}: {exc}"
    #
    trigger_hooks("PostToolUse", block, output)
    return (output)


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Agent Loop
#

def agent_loop(messages:list):

    while (True):

        try:

            response = client.messages.create(
                model      = MODEL,
                system     = SYSTEM,
                messages   = messages,
                tools      = TOOLS,
                max_tokens = 8000,
            )
        #
        except Exception as exc:

            messages.append({
                "role"    : "assistant",
                "content" : [{
                    "type" : "text",
                    "text" : f"[Error] {type(exc).__name__}: {exc}",
                }],
            })
            release_completed_assignment("agent")
            trigger_hooks("Stop", messages)
            return
        #

        messages.append({"role": "assistant", "content": response.content})
        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if (not tool_calls):

            release_completed_assignment("agent")
            trigger_hooks("Stop", messages)
            return
        #

        results = []
        for block in tool_calls:

            print(f"> {block.name}")
            output = execute_tool(block)
            print(output[:300])
            results.append({
                "type"        : "tool_result",
                "tool_use_id" : block.id,
                "content"     : output,
            })
        #
        messages.append({"role": "user", "content": results})
    #
#


def print_last_assistant_message(history:list):

    if (not history):

        return
    #
    for block in history[-1].get("content", []):

        if (getattr(block, "type", None) == "text"):

            print(block.text)
        #
        elif (isinstance(block, dict)  and  block.get("type") == "text"):

            print(block.get("text", ""))
        #
    #
#


def wait_for_cli_event() -> tuple[str, str|None]:

    prompt_visible = False
    while (True):

        if (BUS.peek("lead")):

            if (prompt_visible):

                print()
            #
            return ("wake", None)
        #
        if (not prompt_visible):

            print("s13 >> ", end="", flush=True)
            prompt_visible = True
        #
        readable, _, _ = select.select([sys.stdin], [], [], 0.25)
        if (readable):

            line = sys.stdin.readline()
            if (line == ""):

                return ("quit", None)
            #
            return ("user", line.rstrip("\n"))
        #
    #
#


if (RUN_AGENT_INTERACTIVELY):

    print("s13: agent teams")
    print("Enter a question, press Enter to send. Type q to quit.\n")
    history       = []
    had_teammates = False

    while (True):

        kind, payload = wait_for_cli_event()
        if (kind == "quit"):

            break
        #
        if (kind == "user"):

            if (payload is None  or  payload.strip().lower() in {"q", "exit", ""}):

                break
            #
            trigger_hooks("UserPromptSubmit", payload)
            history.append({"role": "user", "content": payload})
        #
        else:

            inbox = consume_lead_inbox()
            if (not inbox):

                continue
            #
            history.append({
                "role"    : "user",
                "content" : format_team_events(inbox),
            })
            print(f"[wake: {len(inbox)} team event(s) -> new turn]")
        #

        agent_loop(history)
        print_last_assistant_message(history)

        if (active_teammates):

            had_teammates = True
        #
        elif (had_teammates  and  not BUS.peek("lead")):

            print("[all teammates shut down]")
            had_teammates = False
        #
        print()
    #
#
